# 02 - VAE on IMU (lift-level)

Trains a 1D CNN VAE on IMU lifts (N, 400, 96) with a classifier head on the latent z.

- **Input:** `VAE/Tensors/lifts_imu.npz` from notebook 01
- **Split:** 80/20 participant-grouped, sex-stratified
- **Loss:** reconstruction (MSE) + β·KL + λ·CE
- **Output:** model checkpoint, latent embeddings, test predictions, metrics to `VAE/Results/vae_imu/`

In [11]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, confusion_matrix
)

ROOT = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project")
TENSOR_PATH = ROOT / "VAE" / "Tensors" / "lifts_imu.npz"
OUT_DIR = ROOT / "VAE" / "Results" / "vae_imu"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [12]:
# ---- hyperparameters ----
LATENT_DIM = 16
BETA = 1.0          # KL weight
LAMBDA_CE = 1.0     # classifier weight
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-5

LOAD_BINS = np.array([2.3, 4.5, 6.8, 9.1, 11.3])   # kg

In [13]:
# ---- load tensors + metadata ----
data = np.load(TENSOR_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)  # (N, 400, 96)
participant = data["participant"]
sex = data["sex"]
box_mass = data["box_mass"].astype(np.float32)
channel_names = data["channel_names"]

# drop lifts containing any NaN (concentrated in a few subjects with missing sensors)
nan_lift = np.isnan(X).any(axis=(1, 2))
print(f"Dropping {int(nan_lift.sum())} / {len(X)} lifts with NaN")
if nan_lift.any():
    u, c = np.unique(participant[nan_lift], return_counts=True)
    print("  by subject:", dict(zip(u.tolist(), c.tolist())))
keep = ~nan_lift
X = X[keep]
participant = participant[keep]
sex = sex[keep]
box_mass = box_mass[keep]

# map kg -> class index
def mass_to_class(m):
    return int(np.argmin(np.abs(LOAD_BINS - m)))
y = np.array([mass_to_class(m) for m in box_mass], dtype=np.int64)

print("X shape:", X.shape)
print("Subjects:", len(np.unique(participant)))
print("Class dist:", dict(zip(*np.unique(y, return_counts=True))))
print("Sex dist :", dict(zip(*np.unique(sex, return_counts=True))))

Dropping 261 / 4800 lifts with NaN
  by subject: {'P10': 12, 'P16': 9, 'P30': 120, 'P38': 120}
X shape: (4539, 400, 96)
Subjects: 38
Class dist: {0: 912, 1: 900, 2: 912, 3: 903, 4: 912}
Sex dist : {'Female': 2379, 'Male': 2160}


In [14]:
# ---- per-channel normalization (fit on train only, later) ----
# build participant-grouped 80/20 split, sex-stratified via subject-level stratification
subjects = np.unique(participant)
subj_sex = {p: sex[participant == p][0] for p in subjects}

rng = np.random.default_rng(SEED)
test_subjects = []
for s_label in np.unique(list(subj_sex.values())):
    subs_s = np.array([p for p, s in subj_sex.items() if s == s_label])
    rng.shuffle(subs_s)
    n_test = max(1, int(round(0.2 * len(subs_s))))
    test_subjects.extend(subs_s[:n_test].tolist())
test_subjects = set(test_subjects)
train_subjects = set(subjects) - test_subjects

train_mask = np.array([p in train_subjects for p in participant])
test_mask = ~train_mask

print(f"Train subjects: {len(train_subjects)}, Test subjects: {len(test_subjects)}")
print(f"Train lifts: {train_mask.sum()}, Test lifts: {test_mask.sum()}")
print("Test sex dist:", dict(zip(*np.unique([subj_sex[p] for p in test_subjects], return_counts=True))))

Train subjects: 30, Test subjects: 8
Train lifts: 3579, Test lifts: 960
Test sex dist: {'Female': 4, 'Male': 4}


In [15]:
# per-channel z-score (fit on train)
X_train_raw = X[train_mask]
ch_mean = X_train_raw.reshape(-1, X.shape[-1]).mean(axis=0)
ch_std = X_train_raw.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
Xn = (X - ch_mean) / ch_std

X_train = torch.from_numpy(Xn[train_mask]).transpose(1, 2)   # (N, C, T)
X_test = torch.from_numpy(Xn[test_mask]).transpose(1, 2)
y_train = torch.from_numpy(y[train_mask])
y_test = torch.from_numpy(y[test_mask])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: torch.Size([3579, 96, 400]) X_test: torch.Size([960, 96, 400])


In [16]:
# ---- VAE architecture ----
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_c, out_c, kernel_size=k, stride=s, padding=p),
            nn.BatchNorm1d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, op):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose1d(in_c, out_c, kernel_size=k, stride=s, padding=p, output_padding=op),
            nn.BatchNorm1d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class VAE(nn.Module):
    def __init__(self, n_channels, seq_len=400, latent_dim=16, n_classes=5):
        super().__init__()
        # 400 -> 200 -> 100 -> 50
        self.enc = nn.Sequential(
            ConvBlock(n_channels, 64, 7, 2, 3),
            ConvBlock(64, 128, 5, 2, 2),
            ConvBlock(128, 256, 3, 2, 1),
        )
        self.enc_out_len = seq_len // 8  # 50
        self.flat_dim = 256 * self.enc_out_len
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)

        self.fc_dec = nn.Linear(latent_dim, self.flat_dim)
        # 50 -> 100 -> 200 -> 400
        self.dec = nn.Sequential(
            DeconvBlock(256, 128, 3, 2, 1, 1),
            DeconvBlock(128, 64, 5, 2, 2, 1),
            nn.ConvTranspose1d(64, n_channels, 7, 2, 3, output_padding=1),
        )
        self.cls = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(inplace=True),
            nn.Linear(64, n_classes),
        )

    def encode(self, x):
        h = self.enc(x)
        h = h.flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 256, self.enc_out_len)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_rec = self.decode(z)
        logits = self.cls(z)
        return x_rec, mu, logvar, z, logits

model = VAE(n_channels=X.shape[-1], seq_len=X.shape[1], latent_dim=LATENT_DIM, n_classes=len(LOAD_BINS)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

Parameters: 995,205


In [17]:
# ---- training loop ----
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
history = []

def vae_losses(x, x_rec, mu, logvar, logits, y):
    recon = F.mse_loss(x_rec, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    ce = F.cross_entropy(logits, y)
    return recon, kl, ce

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr = {"recon": 0.0, "kl": 0.0, "ce": 0.0, "n": 0, "correct": 0}
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        x_rec, mu, logvar, z, logits = model(xb)
        recon, kl, ce = vae_losses(xb, x_rec, mu, logvar, logits, yb)
        loss = recon + BETA * kl + LAMBDA_CE * ce
        loss.backward()
        opt.step()
        bs = xb.size(0)
        tr["recon"] += recon.item() * bs
        tr["kl"] += kl.item() * bs
        tr["ce"] += ce.item() * bs
        tr["n"] += bs
        tr["correct"] += (logits.argmax(1) == yb).sum().item()

    # test eval
    model.eval()
    te = {"recon": 0.0, "kl": 0.0, "ce": 0.0, "n": 0, "correct": 0}
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            x_rec, mu, logvar, z, logits = model(xb)
            recon, kl, ce = vae_losses(xb, x_rec, mu, logvar, logits, yb)
            bs = xb.size(0)
            te["recon"] += recon.item() * bs
            te["kl"] += kl.item() * bs
            te["ce"] += ce.item() * bs
            te["n"] += bs
            te["correct"] += (logits.argmax(1) == yb).sum().item()

    row = {
        "epoch": epoch,
        "train_recon": tr["recon"] / tr["n"],
        "train_kl": tr["kl"] / tr["n"],
        "train_ce": tr["ce"] / tr["n"],
        "train_acc": tr["correct"] / tr["n"],
        "test_recon": te["recon"] / te["n"],
        "test_kl": te["kl"] / te["n"],
        "test_ce": te["ce"] / te["n"],
        "test_acc": te["correct"] / te["n"],
    }
    history.append(row)
    if epoch % 5 == 0 or epoch == 1:
        print(f"ep{epoch:3d} | tr_rec {row['train_recon']:.3f} kl {row['train_kl']:.3f} ce {row['train_ce']:.3f} acc {row['train_acc']:.3f} "
              f"| te_rec {row['test_recon']:.3f} kl {row['test_kl']:.3f} ce {row['test_ce']:.3f} acc {row['test_acc']:.3f}")

hist_df = pd.DataFrame(history)
hist_df.to_csv(OUT_DIR / "history.csv", index=False)
print("History saved.")

ep  1 | tr_rec 0.997 kl 1.944 ce 1.656 acc 0.196 | te_rec 0.972 kl 0.517 ce 1.618 acc 0.215
ep  5 | tr_rec 0.833 kl 0.132 ce 1.543 acc 0.296 | te_rec 0.827 kl 0.130 ce 1.558 acc 0.279
ep 10 | tr_rec 0.781 kl 0.223 ce 1.331 acc 0.396 | te_rec 0.795 kl 0.212 ce 1.545 acc 0.319
ep 15 | tr_rec 0.764 kl 0.255 ce 1.110 acc 0.501 | te_rec 0.799 kl 0.265 ce 1.983 acc 0.259
ep 20 | tr_rec 0.757 kl 0.286 ce 0.848 acc 0.638 | te_rec 0.780 kl 0.264 ce 2.065 acc 0.271
ep 25 | tr_rec 0.751 kl 0.345 ce 0.661 acc 0.729 | te_rec 0.796 kl 0.270 ce 2.294 acc 0.284
ep 30 | tr_rec 0.747 kl 0.370 ce 0.439 acc 0.833 | te_rec 0.773 kl 0.292 ce 2.766 acc 0.284
ep 35 | tr_rec 0.748 kl 0.352 ce 0.205 acc 0.932 | te_rec 0.774 kl 0.279 ce 3.012 acc 0.314
ep 40 | tr_rec 0.744 kl 0.346 ce 0.188 acc 0.932 | te_rec 0.778 kl 0.268 ce 3.383 acc 0.303
ep 45 | tr_rec 0.745 kl 0.333 ce 0.128 acc 0.957 | te_rec 0.767 kl 0.291 ce 3.830 acc 0.308
ep 50 | tr_rec 0.740 kl 0.324 ce 0.106 acc 0.966 | te_rec 0.771 kl 0.254 ce 3.82

In [18]:
# ---- final evaluation: collect predictions and latents on test set ----
model.eval()
all_z, all_mu, all_logits, all_y = [], [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        mu, logvar = model.encode(xb)
        z = mu  # use mean at inference
        logits = model.cls(z)
        all_mu.append(mu.cpu().numpy())
        all_z.append(z.cpu().numpy())
        all_logits.append(logits.cpu().numpy())
        all_y.append(yb.numpy())

mu_test = np.concatenate(all_mu)
z_test = np.concatenate(all_z)
logits_test = np.concatenate(all_logits)
y_true = np.concatenate(all_y)
y_pred = logits_test.argmax(1)

# regression-like error in kg
mass_true = LOAD_BINS[y_true]
mass_pred = LOAD_BINS[y_pred]

acc = accuracy_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1m = f1_score(y_true, y_pred, average="macro")
mae = mean_absolute_error(mass_true, mass_pred)
rmse = np.sqrt(mean_squared_error(mass_true, mass_pred))

print(f"Test  acc={acc:.3f}  balanced_acc={bal_acc:.3f}  macro_F1={f1m:.3f}  MAE={mae:.3f} kg  RMSE={rmse:.3f} kg")

Test  acc=0.307  balanced_acc=0.307  macro_F1=0.309  MAE=2.412 kg  RMSE=3.208 kg


In [19]:
# ---- fairness: directional error by sex ----
test_sex = sex[test_mask]
signed_err = mass_pred - mass_true   # +ve = overestimate
abs_err = np.abs(signed_err)

fair_rows = []
for s_label in np.unique(test_sex):
    m = test_sex == s_label
    fair_rows.append({
        "sex": s_label,
        "n": int(m.sum()),
        "acc": accuracy_score(y_true[m], y_pred[m]),
        "bal_acc": balanced_accuracy_score(y_true[m], y_pred[m]),
        "mae_kg": float(abs_err[m].mean()),
        "mean_signed_err_kg": float(signed_err[m].mean()),
        "over_rate": float((signed_err[m] > 0).mean()),
        "under_rate": float((signed_err[m] < 0).mean()),
    })
fair_df = pd.DataFrame(fair_rows)
print(fair_df.to_string(index=False))

   sex   n      acc  bal_acc   mae_kg  mean_signed_err_kg  over_rate  under_rate
Female 480 0.320833 0.320833 2.241875           -0.362292   0.310417    0.368750
  Male 480 0.293750 0.293750 2.581250            0.793750   0.445833    0.260417


In [20]:
# ---- save artifacts ----
torch.save({
    "model_state": model.state_dict(),
    "ch_mean": ch_mean, "ch_std": ch_std,
    "latent_dim": LATENT_DIM, "seq_len": X.shape[1], "n_channels": X.shape[-1],
    "channel_names": list(map(str, channel_names)),
}, OUT_DIR / "vae_imu.pt")

np.savez(OUT_DIR / "test_outputs.npz",
         mu=mu_test, z=z_test, logits=logits_test,
         y_true=y_true, y_pred=y_pred,
         mass_true=mass_true, mass_pred=mass_pred,
         participant=participant[test_mask], sex=test_sex)

summary = {
    "acc": acc, "bal_acc": bal_acc, "macro_f1": f1m,
    "mae_kg": float(mae), "rmse_kg": float(rmse),
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    "fairness": fair_rows,
    "config": {
        "latent_dim": LATENT_DIM, "beta": BETA, "lambda_ce": LAMBDA_CE,
        "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR, "seed": SEED,
        "train_subjects": sorted(map(str, train_subjects)),
        "test_subjects": sorted(map(str, test_subjects)),
    },
}
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

fair_df.to_csv(OUT_DIR / "fairness.csv", index=False)
print("Saved to:", OUT_DIR)

Saved to: C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\VAE\Results\vae_imu
